In [14]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [15]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from general_functions.datetime_helper import transform_date_to_timestamp_milliseconds
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [16]:
customer = "Asambeauty"
url = return_api_url()
print(f"url = {url}")
list_workspaces = return_workspace_ids()
workspace_id = [acc["id"] for acc in list_workspaces if acc["name"] == customer]
if len(workspace_id) != 1:
    print(sorted([item["name"] for item in list_workspaces]))
workspace_id = workspace_id[0]

In [17]:
signals = call_api_with_accountId(f"{url}api/signals/query", workspace_id,{},logger=logging.getLogger(__name__))
signals = pd.json_normalize(signals)

In [18]:
model_ids = signals["model"].dropna().unique()
models = call_api_with_accountId(f"{url}api/models/query", workspace_id,{},logger=logging.getLogger(__name__))
models = pd.json_normalize(models)
models = models.rename(columns={"id":"model","type":"model_type","objective":"model_objective",})
len(models)

In [19]:
df = pd.merge(signals,models[["model","model_type","path","model_objective","created"]],on="model",how="left")
df.columns

In [20]:
filtered = df# [signals["connection.platform.name"]=="googleAnalytics"].sort_values(by="name")
#filtered = filtered[filtered["config.purpose"]=="seed"]
filtered["path_date"] = filtered["path"].str.extract(r"/(\d{4}-\d{2}-\d{2})")
filtered[["id","name","status","objective","model_objective","created","path_date","model"]].sort_values(by=["status","name","objective"])

In [21]:
duplicated_names = filtered["name"].value_counts()
duplicated_names

In [22]:
list_ids = ["6a34fb828352ee28a0c1d873","6a34f3c58872c5ad8e9b7af4","6a35160e8c0700b3c35c6ad1","6a3412e5d4da4438025e1816"]
filtered_by_id = signals[signals["id"].isin(list_ids)]
filtered_by_id[["name","connection.platform.name","id","createdAt","status"]]

# Test with audiences

In [23]:
for col in signals.columns:
    for id in list_ids:
        test_id = signals[signals[col].astype("str").str.contains(id)]
        if len(test_id) > 0:
            print(f"{col} contains {id}")

In [24]:
signals.sort_values(by="createdAt",ascending=False)[["id","name","createdAt","status","connection.platform.name"]]